In [1]:
import pandas as pd
import numpy as np
import pickle


In [2]:
with open('model.pkl', 'rb') as f:
    model = pickle.load(f)
with open('train_columns.pkl', 'rb') as f:
    train_columns = pickle.load(f)

In [3]:
test_df = pd.read_csv('game_of_thrones_test.csv', index_col='S.No')

In [4]:
test_df.dropna(axis=1, thresh=10, inplace=True)

In [5]:
test_df['popularity_log'] = np.log10(test_df['popularity'] * 10 + 1)
test_df.drop(columns=['popularity'], inplace=True)

In [6]:
test_df['boolDeadRelations'] = (test_df['numDeadRelations'] > 0).astype(int)
test_df.drop(columns=['numDeadRelations'], inplace=True)

In [7]:
test_df.drop(columns=['dateOfBirth'], inplace=True)

In [8]:
test_df['age_no_data'] = test_df['age'].isna().astype(int)
test_df['age_value'] = test_df['age'].fillna(0)
test_df.drop(columns=['age'], inplace=True)


In [9]:
cultures_grouped = {
    'Old Nations': ['valyrian', 'first men', 'andal', 'andals', 'rhoynar'],
    'the North': ['northmen', 'northern mountain clans', 'crannogmen'],
    'the Iron Islands': ['ironborn', 'ironborn', 'ironmen'],
    'the Mountain and the Vale': ['valemen', 'vale', 'vale mountain clans', 'sistermen'],
    'the Isles and Rivers': ['riverlands', 'rivermen'],
    'the Rock': ['westerman', 'westermen', 'westerlands'],
    'the Stormlands': ['stormlander', 'stormlands'],
    'the Reach': ['reach', 'reachmen', 'the reach'],
    'Dorne': ['dornish', 'dornishmen', 'dorne'],
    'Essos Nations': ['astapor', 'astapori', 'braavosi', 'braavos', 'tyroshi', 'lysene', 'lyseni',
                      'myrish', 'pentoshi', 'qartheen', 'qarth', 'dothraki',
                      'lhazarene', 'lhazareen','meereen', 'meereenese',
                      'norvoshi', 'qohor', 'summer isles', 'summer islands',
                      'summer islander', 'asshai', "asshai'i", 'norvos', 'ghiscari',
                      'ghiscaricari'],
    'Other Nations': ['ibbenese', 'westeros', 'free folk', 'wildling', 'wildlings', 'naathi']
}

In [10]:
culture_to_group = {}
for group, values in cultures_grouped.items():
    for v in values:
        culture_to_group[v] = group
test_df['culture'] = test_df['culture'].str.lower().str.strip().replace(culture_to_group)
test_df['culture'] = test_df['culture'].fillna('culture_no_data')


In [11]:
title_counts = test_df['title'].value_counts()
rare_titles = title_counts[title_counts < 3].index.tolist()
test_df['title'] = test_df['title'].replace(rare_titles, 'Rare')
test_df['title'] = test_df['title'].fillna('unknown_title')


In [12]:
house_counts = test_df['house'].value_counts()
rare_houses = house_counts[house_counts < 3].index.tolist()
test_df['house'] = test_df['house'].replace(rare_houses, 'Rare_house')
test_df['house'] = test_df['house'].fillna('house_unknown')


In [13]:
def spouse_status(row):
    if pd.isna(row['isAliveSpouse']):
        return 'no_spouse'
    elif row['isAliveSpouse'] == 1:
        return 'spouse_alive'
    else:
        return 'spouse_dead'

In [14]:
test_df['spouse_status'] = test_df.apply(spouse_status, axis=1)
spouse_dummies = pd.get_dummies(test_df['spouse_status'], prefix='spouse', dtype=int)
test_df = pd.concat([test_df, spouse_dummies], axis=1)
test_df.drop(columns=['isAliveSpouse', 'spouse_status'], inplace=True)


In [15]:
test_df = pd.concat([test_df, pd.get_dummies(test_df['title'], prefix='title')], axis=1)
test_df = pd.concat([test_df, pd.get_dummies(test_df['culture'], prefix='culture')], axis=1)
test_df = pd.concat([test_df, pd.get_dummies(test_df['house'], prefix='house')], axis=1)


In [16]:
test_df.drop(columns=['title', 'culture', 'house', 'name', 'spouse'], inplace=True, errors='ignore')


In [17]:
test_aligned = test_df.reindex(columns=train_columns, fill_value=0)

In [18]:
pred_test = model.predict(test_aligned)
submission = pd.DataFrame({'isAlive': pred_test}, index=test_df.index)
submission.to_csv('submission.csv')
print("Файл submission.csv успешно создан.")

Файл submission.csv успешно создан.
